<a href="https://colab.research.google.com/github/RobertoSan04/100_days_coding_python/blob/main/hey_score_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hey Score Demo — Par B
## Datathon Hey Banco 2026

## 0. Setup & Imports

In [12]:
# Instalar dependencias
!pip install gradio google-genai --quiet

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
from google import genai
from google.colab import userdata, drive
import warnings
warnings.filterwarnings('ignore')

print("Imports listos")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 5.6 MB/s eta 0:00:00
Imports listos


## 1. Conexión a Google Drive

In [ ]:
drive.mount('/content/drive')

# Rutas base
BASE_PATH = '/content/drive/MyDrive/hey_datathon'
DATA_PATH = f'{BASE_PATH}/data'
OUTPUTS_PATH = f'{BASE_PATH}/outputs'
FIGURES_PATH = f'{BASE_PATH}/figures'

print(f" Drive montado")
print(f" Data: {DATA_PATH}")
print(f" Outputs: {OUTPUTS_PATH}")
print(f" Figures: {FIGURES_PATH}")

Mounted at /content/drive
 Drive montado
 Data: /content/drive/MyDrive/hey_datathon/data
 Outputs: /content/drive/MyDrive/hey_datathon/outputs
 Figures: /content/drive/MyDrive/hey_datathon/figures


## 2. Carga de Datos

In [7]:
# Carga de datos
clientes = pd.read_csv(f'{DATA_PATH}/hey_clientes.csv')
productos = pd.read_csv(f'{DATA_PATH}/hey_productos.csv')

print(f"clientes: {clientes.shape}")
print(f"productos: {productos.shape}")

clientes: (15025, 22)
productos: (38909, 13)


In [22]:
# Cargar archivo master_df
master_df = pd.read_parquet(f'{OUTPUTS_PATH}/master_df.parquet')
score_per_user = pd.read_csv(f'{OUTPUTS_PATH}/score_per_user.csv')

print(f"master_df: {master_df.shape}")
print(f"columnas: {list(master_df.columns)}")

master_df: (15025, 79)
columnas: ['user_id', 'edad', 'genero', 'estado', 'ciudad', 'nivel_educativo', 'ocupacion', 'ingreso_mensual_mxn', 'antiguedad_dias', 'es_hey_pro', 'nomina_domiciliada', 'canal_apertura', 'score_buro', 'dias_desde_ultimo_login', 'preferencia_canal', 'satisfaccion_1_10', 'recibe_remesas', 'usa_hey_shop', 'idioma_preferido', 'tiene_seguro', 'num_productos_activos', 'patron_uso_atipico', 'antiguedad_anios', 'login_recency_score', 'es_joven', 'n_credito_auto', 'n_credito_nomina', 'n_credito_personal', 'n_cuenta_debito', 'n_cuenta_negocios', 'n_inversion_hey', 'n_seguro_compras', 'n_seguro_vida', 'n_tarjeta_credito_garantizada', 'n_tarjeta_credito_hey', 'n_tarjeta_credito_negocios', 'tiene_inversion', 'tiene_credito', 'tiene_cuenta_negocios', 'utilizacion_promedio', 'utilizacion_max', 'deuda_total', 'limite_total', 'num_productos_credito', 'monto_invertido', 'gasto_total', 'gasto_promedio', 'variabilidad_gasto', 'num_txn_total', 'num_compras', 'num_pagos_credito', 'nu

## 3. Mock de Usuario (placeholder hasta que Par A entregue master_df)

In [8]:
# Mock de un usuario hardcodeado
# Se reemplaza en Fase 3 cuando Par A entregue master_df.parquet

mock_usuario = {
    "user_id": "USR-00042",
    "hey_score": 67,
    "dim_salud_financiera": 72,
    "dim_engagement": 65,
    "dim_credito": 58,
    "dim_inversion": 45,
    "dim_conversacional": 71,
    "segmento": "Cliente consolidado",
    "tema_dominante": "pagos",
    "sentimiento_promedio": 0.2
}

print("Mock de usuario definido:")
for k, v in mock_usuario.items():
    print(f"  {k}: {v}")

Mock de usuario definido:
  user_id: USR-00042
  hey_score: 67
  dim_salud_financiera: 72
  dim_engagement: 65
  dim_credito: 58
  dim_inversion: 45
  dim_conversacional: 71
  segmento: Cliente consolidado
  tema_dominante: pagos
  sentimiento_promedio: 0.2


In [23]:
def get_usuario(user_id):
    row = master_df[master_df['user_id'] == user_id].iloc[0]
    return {
        "user_id": row['user_id'],
        "hey_score": round(row['hey_score'], 1),
        "dim_solidez": round(row['dim_solidez'], 1),
        "dim_diversificacion": round(row['dim_diversificacion'], 1),
        "dim_gasto": round(row['dim_gasto'], 1),
        "dim_engagement": round(row['dim_engagement'], 1),
        "dim_proteccion": round(row['dim_proteccion'], 1),
        "segmento": row['cluster_etiqueta'],
        "tema_dominante": row['tema_dominante'],
        "sentimiento_promedio": round(row['sentimiento_promedio'], 2),
        "edad": int(row['edad']),
        "ocupacion": row['ocupacion'],
        "ingreso_mensual_mxn": int(row['ingreso_mensual_mxn'])
    }

# Prueba con un usuario real
usuario = get_usuario('USR-00042')
print(usuario)

{'user_id': 'USR-00042', 'hey_score': np.float64(nan), 'dim_solidez': np.float64(nan), 'dim_diversificacion': np.float64(4.4), 'dim_gasto': np.float64(13.4), 'dim_engagement': np.float64(11.5), 'dim_proteccion': np.float64(nan), 'segmento': '🌱 Recién bancarizado', 'tema_dominante': 'credito', 'sentimiento_promedio': np.float64(0.0), 'edad': 34, 'ocupacion': 'Empleado', 'ingreso_mensual_mxn': 18000}


## 4. API de Gemini — Test de conexión

In [21]:
import google.generativeai as genai
from google.colab import userdata

# Fetch the actual key value from Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')

genai.configure(api_key=api_key)

# The model identifier 'gemma-4-31b-it' is correct for the 31B instruction-tuned version
model = genai.GenerativeModel('gemma-4-31b-it')

response = model.generate_content("Hello Gemma 4, how are you today?")
print(response.text)

*   User's input: "Hello Gemma 4, how are you today?"
    *   Intent: Greeting and social check-in.
    *   Entity: Addressing me as "Gemma 4".

    *   Name: Gemma 4.
    *   Developer: Google DeepMind.
    *   Nature: Large Language Model, open weights.
    *   Capabilities: Text/image input, text output.
    *   Tone: Helpful, polite, professional.

    *   Standard AI response: AI doesn't have feelings, but can simulate a polite and positive state of being.
    *   Acknowledgement: Acknowledge the greeting and the name.
    *   Call to action: Ask how to help the user.

    *   *Draft 1:* Hello! I'm doing well, thank you for asking. How can I help you today?
    *   *Draft 2:* Hello! As an AI, I don't have feelings, but I'm functioning perfectly and ready to assist. What's on your mind?
    *   *Refining for persona:* Keep it friendly and helpful. "I'm doing great, thank you for asking! How can I help you today?"
I'm doing great, thank you for asking! How can I help you today?


## 5. Clustering K-Means

## 6. Demo Gradio

## 7. Counterfactual